# Individual Income Tax Overview, Dependents, and Filing Status

**Selected Excercises**

McGraw Hill's Taxation of Individuals and Business Entities 2025

Spilker, Ayers, Lewis, Weaver, Barrick, Robinson, Worsham

In [ ]:
import os
import sys
import pandas as pd
from datetime import date
from IPython.display import display, HTML

mod_path = os.path.abspath(os.path.join(".."))
if mod_path not in sys.path:
    sys.path.append(mod_path)

from stackCalc import RPNEngine
from income_tax import OrdinaryIncomeTax, PreferentialIncomeTax, IncomeTaxSummary, DependencyTest

---

### **Jeremy (unmarried) earned 100,000 in salary and 6,000 in interest income during the year. Jeremy’s employer withheld 10,000 of federal income taxes from Jeremy’s paychecks during the year. Jeremy has one qualifying dependent child (age 14) who lives with him. Jeremy qualifies to file as head of household and has 25,000 in itemized deductions.**

In [ ]:
status = "HOH"               # Head of Household
age = 0                      # Not applicable
salary = 100000              # Ordinary income
interest = 6000              # Ordinary income
adjustments = 0              # Above the line adjustments
itemized_deduction = 25000   # Below the line adjustments
qbi_deduction = 0
tax_withholdings = 10000     # Tax Prepayments
child_tax_credit = 2200      # CTC - Child Tax Credit 2026


**- a) Determine Jeremy’s tax refund or taxes due.**

In [ ]:
gross_income = salary + interest
it = OrdinaryIncomeTax(status, age, gross_income, adjustments, itemized_deduction, qbi_deduction).calculate_tax()
capital_gains = 0
capital_gains_tax = 0
tax = IncomeTaxSummary(gross_income,
                              it["adjustments"],  
                              it["deduction"], 
                              qbi_deduction,
                              capital_gains,
                              it["ordinary_tax"],
                              capital_gains_tax,
                              -child_tax_credit,
                              -tax_withholdings).income_tax_summary()

display(HTML(tax.to_html(index=False))) 

**- b) Assume that in addition to the original facts, Jeremy has a long-term capital gain of 4,000. What is Jeremy’s tax refund or tax due including the tax on the capital gain?**

In [ ]:
capital_gains = 4000
it = OrdinaryIncomeTax(status, age, gross_income, adjustments, itemized_deduction, qbi_deduction).calculate_tax()
cgt = PreferentialIncomeTax(status, age, gross_income, 0, capital_gains).calculate_tax()
capital_gains_tax = cgt[1]

tax = IncomeTaxSummary(gross_income,
                              it["adjustments"],  
                              it["deduction"], 
                              qbi_deduction,
                              capital_gains,
                              it["ordinary_tax"],
                              capital_gains_tax,
                              -child_tax_credit,
                              -tax_withholdings).income_tax_summary()

display(HTML(tax.to_html(index=False))) 

- **c) Assume the original facts except that Jeremy has only 7,000 in itemized deductions. What is Jeremy’s tax refund or tax due?**

In [ ]:
itemized_deduction = 7000
it = OrdinaryIncomeTax(status, age, gross_income, adjustments, itemized_deduction, qbi_deduction).calculate_tax()
capital_gains = 0
capital_gains_tax = 0
tax = IncomeTaxSummary(gross_income,
                              it["adjustments"], 
                              it["deduction"], 
                              qbi_deduction,
                              capital_gains,
                              it["ordinary_tax"],
                              capital_gains_tax,
                              -child_tax_credit,
                              -tax_withholdings).income_tax_summary()

display(HTML(tax.to_html(index=False))) 

---

### **Aram’s taxable income before considering capital gains and losses is 60,000. Determine Aram’s taxable income and how much of the income will be taxed at ordinary rates in each of the following alternative scenarios (assume Aram files as a single taxpayer).**

In [ ]:
status = "SNG"
age = 0                      # Not applicable
income = 60000               # Ordinary income
adjustments = 0              # Above the line adjustments
tax_withholdings = 0         # Tax Prepayments
child_tax_credit = 0         # CTC - Child Tax Credit 2026

lt_cap_gains = 5000
lt_cap_losses = 500
st_cap_gains = 1200
st_cap_losses = 900

- **a) Aram sold a capital asset that he owned for more than one year for a 5,000 gain, a capital asset that he owned for more than one year for a 500 loss, a capital asset that he owned for six months for a 1,200 gain, and a capital asset he owned for two months for a 900 loss.**

In [ ]:
# Netting process
lt_net = lt_cap_gains - lt_cap_losses
st_net = st_cap_gains - st_cap_losses

print(f"""
Aram's taxable income:

Taxable income:            {income:,.0f}
Short-term capital gains   {st_net:,.0f}
Long-term capital gains    {lt_net:,.0f}
                           ------
Taxable income             {income + st_net + lt_net:,.0f}

Aram's taxable income that will be taxed at ordinary rates:
      
Taxable income:            {income:,.0f}
Short-term capital gains   {st_net:,.0f}
                           ------
Ordinary income            {income + st_net:,.0f}
    """)

- **b) Aram sold a capital asset that he owned for more than one year for a 2,000 gain, a capital asset that he owned for more than one year for a 2,500 loss, a capital asset that he owned for six months for a 200 gain, and a capital asset he owned for two months for a 1,900 loss.**

In [ ]:
lt_cap_gains = 2000
lt_cap_losses = 2500
st_cap_gains = 200
st_cap_losses = 1900

# Netting process
lt_net = lt_cap_gains - lt_cap_losses
st_net = st_cap_gains - st_cap_losses

capital_loss_limit = min(abs(lt_net) + abs(st_net), 3000)


print(f"""
Netting Process:
Net long-term capital loss    {abs(lt_net):,.0f}
Net short-term capital loss   {abs(st_net):,.0f}
                              ------
Total capital loss            {abs(lt_net) + abs(st_net):,.0f}""")

print(f"""
Aram's taxable income:

Taxable income:               {income:,.0f}
Capital loss                  {capital_loss_limit:,.0f}
                              ------
Taxable income                {income - capital_loss_limit:,.0f}

Note: In this case the whole taxable income will be taxed at ordinary rates.""")


- **c) Aram sold a capital asset that he owned for more than one year for a 2,500 loss, a capital asset that he owned for six months for a 4,200 gain, and a capital asset he owned for two months for a 300 loss.**

In [ ]:
lt_cap_gains = 0
lt_cap_losses = 2500
st_cap_gains = 4200
st_cap_losses = 300

# Netting process
lt_net = lt_cap_gains - lt_cap_losses
st_net = st_cap_gains - st_cap_losses
net_capital_gain = lt_net + st_net

print(f"""
Netting Process:
Net long-term capital loss            {lt_net:,.0f}
Net short-term capital gain            {st_net:,.0f}
                                      ------
Net short-term capital gain            {net_capital_gain:,.0f}""")

print(f"""
Aram's taxable income:

Taxable income:                       {income:,.0f}
Net short-term capital gain           {net_capital_gain:,.0f}
                                      ------
Taxable income                        {income + net_capital_gain:,.0f}

Note: In this case the whole taxable income will be taxed at ordinary rates.""")

- **d) Aram sold a capital asset that he owned for more than one year for a 3,000 gain, a capital asset that he owned for more than one year for a 300 loss, a capital asset that he owned for six months for a 200 gain, and a capital asset he owned for two months for a 1,900 loss.**

In [ ]:
lt_cap_gains = 3000
lt_cap_losses = 300
st_cap_gains = 200
st_cap_losses = 1900

# Netting process
lt_net = lt_cap_gains - lt_cap_losses
st_net = st_cap_gains - st_cap_losses
net_capital_gain = lt_net + st_net

print(f"""
Netting Process:
Net long-term capital loss            {lt_net:,.0f}
Net short-term capital gain          {st_net:,.0f}
                                      ------
Net short-term capital gain           {net_capital_gain:,.0f}

Aram's taxable income:

Taxable income:                     {income:,.0f}
Net Long-term capital gains          {net_capital_gain:,.0f}
                                    ------
Taxable income                      {income + net_capital_gain:,.0f}

Aram's taxable income that will be taxed at ordinary rates:
      
Taxable income:                     {income:,.0f}
                                    ------
Ordinary income                     {income:,.0f}
    """)


---

### **David and Lilly Fernandez have determined their tax liability on their joint tax return to be 3,100. They have made prepayments of 1,900 and also have a child tax credit of 2,000. What is the amount of their tax refund or taxes due?**

In [ ]:
tax_liability = 3100
prepayments = 1900
child_tax_credit = 2000
tax_due_refund = tax_liability - prepayments - child_tax_credit
due_or_refund = "Tax refund" if tax_due_refund < 0 else "Tax due"
print(f"""
Tax liability       {tax_liability:,.0f}
Prepayments         {prepayments:,.0f}
Child Tax Credit    {child_tax_credit:,.0f}
                    -----
{due_or_refund}           {abs(tax_due_refund)}""")


---

### **Ekiya, who is single, has been offered a position as a city landscape consultant. The position pays 125,000 in wages. Assume Ekiya has no dependents. Ekiya deducts the standard deduction instead of itemized deductions, and she is not eligible for the qualified business income deduction.**

In [ ]:
status = "SNG"               # Head of Household
age = 0                      # Not applicable
job1_wages = 125000          # Ordinary income
adjustments = 0              # Above the line adjustments
itemized_deduction = 0       # Below the line adjustments
qbi_deduction = 0
job1_tax = OrdinaryIncomeTax(status, age, job1_wages, adjustments, itemized_deduction, qbi_deduction).calculate_tax()["ordinary_tax"]
job1_after_tax_compensation = job1_wages - job1_tax

- a) **What is the amount of Ekiya’s after-tax compensation (ignore payroll taxes)?**

In [ ]:

job1_tax = OrdinaryIncomeTax(status, age, job1_wages, adjustments, itemized_deduction, qbi_deduction).calculate_tax()["ordinary_tax"]
job1_after_tax_compensation = job1_wages - job1_tax

print(f"""
Wages                     {job1_wages:,.0f}
Tax                        {job1_tax:,.0f}
                          --------
After-tax Compensation    {job1_after_tax_compensation:,.0f}

Note: Standard deduction for a single tax payer in 2026 is: $16,100""")


- **b) Suppose Ekiya receives a competing job offer of 120,000 in wages and nontaxable (excluded) benefits worth 5,000. What is the amount of Ekiya’s after-tax compensation for the competing offer? Which job should she take if taxes are the only concern?**

In [ ]:
job2_wages = 120000
non_taxable_benefits = 5000
compensation = job2_wages + non_taxable_benefits
job2_tax = OrdinaryIncomeTax(status, age, job2_wages, adjustments, itemized_deduction, qbi_deduction).calculate_tax()["ordinary_tax"]
job2_after_tax_compensation = compensation - job2_tax

comparison = {
    "": ["Wages", "Non Taxable Compensation", "Tax", "After Tax Compensation"],
    "Job 1": [job1_wages, 0, job1_tax, job1_after_tax_compensation],
    "Job 2": [job2_wages, non_taxable_benefits, job2_tax, job2_after_tax_compensation],
}

df = pd.DataFrame(comparison)
df["Job 1"] = df["Job 1"].map("${:,.0f}".format)
df["Job 2"] = df["Job 2"].map("${:,.0f}".format)
display(HTML(df.to_html(index=False)))

print(f"""
Ekiya should choose Job 2 which offers the best after tax compensation along with additional benefits.

Note: Standard deduction for a single tax payer in 2026 is: $16,100
""")

---

### **Through November, Cameron has received gross income of 120,000. For December, Cameron is considering whether to accept one more work engagement for the year. Engagement 1 will generate 7,000 of revenue at a cost to Cameron of 3,000, which is deductible for AGI. In contrast, engagement 2 will generate 5,000 of qualified business income (QBI), which is eligible for the 20 percent QBI deduction. Cameron files as a single taxpayer.**

- **a) Calculate Cameron’s taxable income assuming he chooses engagement 1 and assuming he chooses engagement 2. Assume he has no itemized deductions.**

In [ ]:
qbi_pct = .20
gross_income = 120000
standard_deduction = 16100

# Scenario 1
engagement_1_income = gross_income + 7000
engagement_1_cost = 3000 # for agi
engagement_1_agi = engagement_1_income - engagement_1_cost
engagement_1_ti = engagement_1_agi - standard_deduction

# Scenario 2
engagement_2_income = gross_income + 5000
engagement_2_qbi = 5000 * qbi_pct
engagement_2_agi = engagement_2_income - 0
engagement_2_ti = engagement_2_agi - engagement_2_qbi- standard_deduction

data = {
    "Items": ["Gross income", "Adjustments", "AGI", "Standard Deduction", "QBI_Deduction", "Taxable Income"],
    "Engagement 1": [engagement_1_income, engagement_1_cost, engagement_1_agi, standard_deduction, 0, engagement_1_ti],
    "Engagement 2": [engagement_2_income, 0, engagement_2_agi, standard_deduction, engagement_2_qbi, engagement_2_ti]
}

df = pd.DataFrame(data)
df["Engagement 1"] = df["Engagement 1"].map("{:,.0f}".format)
df["Engagement 2"] = df["Engagement 2"].map("{:,.0f}".format)
display(HTML(df.to_html(index=False)))


- b) **Which engagement maximizes Cameron’s after-tax cash flow? Explain.**

In [ ]:
status = "SNG"
age = 0            # Not applicable
engagement_1_tax = OrdinaryIncomeTax(status, age, engagement_1_income, engagement_1_cost, 0, 0).calculate_tax()["ordinary_tax"]
engagement_2_tax = OrdinaryIncomeTax(status, age, engagement_2_income, 0, 0, engagement_2_qbi).calculate_tax()["ordinary_tax"]

engagement_1_after_tax = engagement_1_agi - engagement_1_tax
engagement_2_after_tax = engagement_2_income - engagement_2_tax

data = {
    "Items": ["Gross income", "Adjustments", "AGI", "Standard Deduction", "QBI_Deduction", "Taxable Income", "Tax", "After-tax income"],
    "Engagement 1": [engagement_1_income, engagement_1_cost, engagement_1_agi, standard_deduction, 0, engagement_1_ti, engagement_1_tax, engagement_1_after_tax],
    "Engagement 2": [engagement_2_income, 0, engagement_2_agi, standard_deduction, engagement_2_qbi, engagement_2_ti, engagement_2_tax, engagement_2_after_tax]
}

df = pd.DataFrame(data)
df["Engagement 1"] = df["Engagement 1"].map("{:,.0f}".format)
df["Engagement 2"] = df["Engagement 2"].map("{:,.0f}".format)
display(HTML(df.to_html(index=False)))

print("""Engagement 2 maximizes Cameron's after-tax cash flow ($106,506 vs $105,506).  Even though both engagements produce
the same taxable income and the same tax liability ($18,494), they differ in the actual cash flow.
      
    - Engagement 1's $3,000 deduction represents a real cash expense that reduces the cash Cameron actually keeps.
    - Engagement 2's $1,000 QBI deduction is a free tax benefit - it lowers his tax liability without costing him
      any cash out of pocket.""")


---


### **Nitai, who is single and has no dependents, was planning on spending the weekend repairing his car. On Friday, Nitai’s employer called and offered him 500 in overtime pay if he would agree to work over the weekend. Nitai could get his car repaired over the weekend at Autofix for 400. If Nitai works over the weekend, he will have to pay the 400 to have his car repaired, but he will earn 500. Assume Nitai’s marginal tax rate is 12 percent.**

In [ ]:
auto_fix = 400
overtime_pay = 500
marginal_tax_rate = .12


- **a) Strictly considering tax factors, should Nitai work or repair his car if the 400 he must pay to have his car fixed is not deductible?**

In [ ]:
after_tax = overtime_pay * (1 - marginal_tax_rate)
net_cash_benefit = after_tax - auto_fix

data = {
    "Description": ["Taxable Income", "After-tax", "Auto Fix", "Net Cash Benefit"],
    "Scenario 1 (Work)": [overtime_pay, after_tax, auto_fix, net_cash_benefit],
    "Scenario 2 (Don't work)": [0 , 0, 0, 0]
}

df = pd.DataFrame(data)
df["Scenario 1 (Work)"] = df["Scenario 1 (Work)"].map("{:,.0f}".format)
df["Scenario 2 (Don't work)"] = df["Scenario 2 (Don't work)"].map("{:,.0f}".format)
display(HTML(df.to_html(index=False)))
print("In this scenario, if Nitai goes to work he gets a $40 Net Cash Benefit, since $40 > $0 he should choose this option.")


- **b) Strictly considering tax factors, should Nitai work or repair his car if the 400 he must pay to have his car fixed is deductible for AGI?**

In [ ]:
taxable_income = overtime_pay - auto_fix
after_tax = taxable_income * (1 - marginal_tax_rate)
net_cash_benefit = after_tax 

data = {
    "Description": ["Income", "Auto Fix", "Taxable Income", "After-tax", "Net Cash Benefit"],
    "Scenario 1 (Work)": [overtime_pay, auto_fix, taxable_income, after_tax, net_cash_benefit],
    "Scenario 2 (Don't work)": [0 , 0, 0, 0, 0]
}

df = pd.DataFrame(data)
df["Scenario 1 (Work)"] = df["Scenario 1 (Work)"].map("{:,.0f}".format)
df["Scenario 2 (Don't work)"] = df["Scenario 2 (Don't work)"].map("{:,.0f}".format)
display(HTML(df.to_html(index=False)))
print("""In this scenario, if Nitai goes to work he gets a $88 Net Cash Benefit, since $88 > $0 he should choose this option.""")


---

### **Rank the following three single taxpayers in order of the magnitude of taxable income (from lowest to highest) and explain your results. Assume none of the taxpayers contributed to charity this year.**

| | Ahmed | Baker | Chin |
| --- | :---: | :---: | :---: |
| Gross income | 90,000 | 90,000 | 90,000 |
| Deductions for AGI | 14,000 | 7,000 | 0 |
| Itemized deductions | 0 | 7,000 | 18,000 |
| Qualified Business Income Deduction | 0 | 2,000 | 10,000 |

In [ ]:
data = {
    "Ahmed": ["SNG", 0, 90000, 14000, 0, 0],
    "Baker": ["SNG", 0, 90000, 7000, 7000, 2000],
    "Chin": ["SNG", 0, 90000, 0, 18000, 10000]
}

rankings = { "Ahmed": [], "Baker": [], "Chin": [] }

# Take the larger of standard deduction or itemized deduction
standard_deduction = 16100 # Single 2026
for k, v in data.items():
    deduction = standard_deduction if standard_deduction >= v[4] else v[4]
    v[4] = deduction

for k, v in data.items():
    taxable_income = v[2] - v[3] - v[4] - v[5]
    rankings[k].append(taxable_income)

df = pd.DataFrame(rankings)
df = df.transpose()
df = df.sort_values(by=0)
df = df.rename(columns={0: "Taxable income"})
df.index.name = "Name"
df = df.reset_index()
df["Taxable income"] = df["Taxable income"].map("{:,.0f}".format)

display(HTML(df.to_html(index=False)))

print("""Taxable income differences come from the interaction of deductions for AGI,
the greater of itemized or standard deduction, and QBI deductions. Ahmed and Baker both
end up using the standard deduction despite having some itemized deductions, while Chin's 
itemized deductions exceed the standard, giving Chin the largest from AGI deduction relative to
gross income.""")

---

### **Aishwarya’s husband passed away in 2023. She needs to determine whether Jasmine, her 17-year-old stepdaughter, who is single, qualifies as her dependent in 2024. Jasmine is a resident but not a citizen of the United States. She lived in Aishwarya’s home from June 15 through December 31, 2024. Aishwarya provided more than half of Jasmine’s support for 2024.**



- **a) Is Aishwarya allowed to claim Jasmine as a dependent for 2024?**

In [ ]:
taxpayer_age = 40                      # Not given, for computation purposes
dependent_age = 17                     # As given
start_date = date(2024, 6, 15)         # June 15, 2024 
end_date = date(2024, 12, 31)          # December 31, 2024                                                              

dependency_test = DependencyTest()
core_dependency = dependency_test.core_dependency_test(
    dependent_taxpayer_test=True,      # Stepchild
    citizenship_residency_test=True,   # A resident of the US
    joint_return_test=True             # Not given, seems reasonable
    )

qualifying_child = dependency_test.qualifying_child_test(
    relationship_test=True,                                                 # Stepchild
    age_test=dependent_age < 19 and dependent_age < taxpayer_age,           # 17 < 19 and 17 < 40 = True
    residence_test= dependency_test.residence_tests(start_date, end_date),  # June 15 - December 31, 2024
    half_support_test=True                                                  # True per text
    )
print(core_dependency[1])
print(qualifying_child[1])
print("""\nConclusion: Aishwarya may claim Jasmine as her dependent for 2024 as a qualifying child.

Jasmine satisfies the citizen/resident, joint-return, relationship, age, residency, and support requirements
based on the facts provided.  The fact that Jasmine is a noncitizen does not prevent the dependency claim because
she is a U.S resident.""")


- **b) Would Aishwarya be allowed to claim Jasmine as a dependent for 2024 if Aishwarya provided more than half of Jasmine’s support in 2024, Jasmine lived in Aishwarya’s home from July 15 through December 31 of 2024, and Jasmine reported gross income of 7,000 for the year?**



In [ ]:
start_date = date(2024, 7, 15)
end_date = date(2024, 12, 31)

core_dependency = dependency_test.core_dependency_test(
    dependent_taxpayer_test=True,       # stepchild
    citizenship_residency_test=True,    # a resident of the us
    joint_return_test=True              # not given, seems reasonable
    )

qualifying_child = dependency_test.qualifying_child_test(
    relationship_test=True,                                                 # stepchild
    age_test=dependent_age < 19 and dependent_age < taxpayer_age,           # 17 < 19 and 17 < 40 = true
    residence_test= dependency_test.residence_tests(start_date, end_date),  # june 15 - december 31, 2024
    half_support_test=True)                                                 # true per text

print(core_dependency[1])
print(qualifying_child[1])
print(f"""\naishwarya is not allowed to claim jasmine as her dependent as a qualifying child for 2024
because she fails the residence test. but she may be able to claim her as a qualifying relative, let's see:""")

jasmine_gross_income = 7000
gross_income_limit = 5050 # 2024
qualifying_relative = dependency_test.qualifying_relative_test(
    relationship_test=True, # Stepchild
    support_test=True, #    # More than half 
    gross_income_test=jasmine_gross_income < gross_income_limit)
print(qualifying_relative[1])
print(f"""\nAnswer: Aishwarya may not claim Jasmine as a dependent for 2024. Jasmine fails the qualifying-child
residence test because she lived with Aishwaryia for less than half of 2024. She also cannot qualify 
as a qualifying relative because her $7,000 gross income exceeds the 2024 limit of $5,050.""")

- **c) Would Aishwarya be allowed to claim Jasmine as a dependent for 2024 if Aishwarya provided more than half of Jasmine’s support in 2024, Jasmine lived in Aishwarya’s home from July 15 through December 31 of 2024, and Jasmine reported gross income of 2,500 for the year?**

In [ ]:
jasmine_gross_income = 2500
gross_income_limit = 5050 # 2024
qualifying_relative = dependency_test.qualifying_relative_test(relationship_test=True, # Stepchild
                                                               support_test=True, #    # More than half 
                                                               gross_income_test=jasmine_gross_income < gross_income_limit)

print(f"""Jasmine won't qualify as qualifying child because she will fail the residence test as before,
however, Aishwarya may be able to claim Jasmine as a qualifying relative.  Let's find out.""")

print(qualifying_relative[1])
print(f"""\nAnswer: Indeed Aishwarya is allowed to claim Jasmine as a qualifying relative for 2024.""")

---

##### **The Samsons are trying to determine whether they can claim their 22-year-old adopted son, Jason, as a dependent. Jason is currently a full-time student at an out-of-state university. Jason lived in his parents’ home for three months of the year, and he was away at school for the rest of the year. He received 9,500 in scholarships this year for his outstanding academic performance and earned 4,800 of income working a part-time job during the year. The Samsons paid a total of 5,000 to support Jason while he was away at college. Jason used the scholarship, the earnings from the part-time job, and the money from the Samsons as his only sources of support.**

In [ ]:
# Data Provided
dependent_age = 22         # Given
parents_age = 46           # Not given, needed for logic, assumes parents are older.
full_time_student = True   # Given

# Support Summary
scholarships = 9500       # Scholarships are excluded from support test
own_support = 4800
parents_support = 5000

# Residency period
# Note A: Reg. §1.152-1(b) - Time that a child or the taxpayer is temporarily away from the 
# taxpayer’s home because the child or taxpayer is ill, is pursuing an education, or has other 
# special circumstances is counted as though the child or taxpayer were living in the taxpayer’s home.

start_date = date(2024, 1, 1)
end_date = date(2024, 12, 31)



- **a) Can the Samsons claim Jason as their dependent?**

In [ ]:
core_dependency = dependency_test.core_dependency_test(
    dependent_taxpayer_test=True,       # Son
    citizenship_residency_test=True,    # Not given, assuming True
    joint_return_test=True              # Not given, assuming True
    )

qualifying_child = dependency_test.qualifying_child_test(
    relationship_test=True,                                                              # Son
    age_test=dependent_age < 24 and dependent_age < parents_age and full_time_student,   # 22 < 24 and 22 < 45 and full_time_student = true
    residence_test= dependency_test.residence_tests(start_date, end_date),               # See Note A.
    half_support_test=parents_support > own_support)                                     # True as given

print(core_dependency[1])
print(qualifying_child[1])
print(f"""\nAnswer: The Samsons may claim Jason as a dependent. They can claim him as a qualifying child because Jason satisfies 
the citizen/resident, joint-return, relationship, age, residency, and support requirements based on the facts provided. """)

- **b) Assume the original facts except that Jason’s grandparents, not the Samsons, provided Jason with the 5,000 worth of support. Can the Samsons (Jason’s parents) claim Jason as their dependent? Why or why not?**


In [ ]:
grandparents_support = 5000
support_test = own_support < grandparents_support

print(f"""
Answer:  
The tax code states that a qualifying child must not have provided more than half of their own support.
It does not matter who provides the rest. Did Jason provide more than half of his own support? Let's see:

Jason's own support:    {own_support:,.0f}
Grandparent's support:  {grandparents_support:,.0f}
Support test:           {support_test}

As we can see, Jason did not provide more than half of his own support.  It doesn't matter that the $5,000
came from his grandparents instead of the Samsons.  The support test for a qualifying child only looks at
Jason's own contribution, not the source fo the remaining support.  So yes, Jason can still be claimed as a
qualifying child dependent by the Samsons.""")

- **c) Assume the original facts except substitute Jason’s grandparents for his parents. Determine whether Jason’s grandparents can claim Jason as a dependent.**


In [ ]:
print("""Answer:
Assuming everything but the relationship has changed from the original facts, Jason's grandparents may claim Jason
as a qualifying child dependent.  
      
Here is why:
A qualifying child must be an eligible relative of the taxpayer.  Elible relatives include the taxpayer's child or a
descendent of a child, so a grandchild qualifies.  The other tests (age, residence, support) are unaffected by the 
change in relationship, so they still pass as well.  Jason can therefore still be claimed a qualifying child dependent,
now by his grandparents.""")

- **d) Assume the original facts except that Jason earned $6,500 while working part time and used this amount for his support. Can the Samsons claim Jason as their dependent? Why or why not?**

In [ ]:
own_support = 6500
support_test = own_support < parents_support

print(f"""
Answer:  
The tax code states that a qualifying child must not have provided more than half of their own support.
Did Jason provide more than half of his own support? Let's see:

Jason's own support:    {own_support:,.0f}
Parent's support:       {parents_support:,.0f}
Support test:           {support_test}

Jason did provide more than half of his own support, so he fails the qualifying child test. He also fails
as a qualifying relative, since the parents didn't provide more than half of his support either. The results
are that the Samsons may not claim may not claim Jason as either a qualifying child or a qualifying relative.""")

---

##### **John and Tara Smith are married and have lived in the same home for over 20 years. John’s uncle Tim, who is 64 years old, has lived with the Smiths since March of this year. Tim is searching for employment but has been unable to find any—his gross income for the year is 2,000. Tim used all 2,000 toward his own support. The Smiths provided the rest of Tim’s support by providing him with lodging valued at 5,000 and food valued at 2,200.**

In [ ]:
# age
dependent_age = 64 # Given
taxpayer_age = 46  # Not given, seems reasonable

# Income
tim_own_support = 2000
provided_lodging = 5000
provided_food = 2200
provided_support = provided_lodging + provided_food
gross_income_limit = 5050 # 2024

start_date = date(2024, 3, 1)
end_date = date(2024, 12, 31)

- **a) Are the Smiths able to claim Tim as a dependent?**

In [ ]:
core_dependency = dependency_test.core_dependency_test(
    dependent_taxpayer_test=True,       # A sibling of the taxpayer's mother of father
    citizenship_residency_test=True,    # Not given, assuming True
    joint_return_test=True              # Not given, assuming True
    )

qualifying_child = dependency_test.qualifying_child_test(
    relationship_test=False,                                                 # Not a descendant, sibling or descendant of sibling
    age_test=dependent_age < 19 and dependent_age < taxpayer_age,            # 64 < 19 and 64 < 46 = false
    residence_test= dependency_test.residence_tests(start_date, end_date),   # Given
    half_support_test=own_support < provided_support)                        # True as given

qualifying_relative = dependency_test.qualifying_relative_test(
    relationship_test=True, # A sibling of the taxpayer's mother of father
    support_test=provided_support > own_support, #    # More than half 
    gross_income_test=tim_own_support < gross_income_limit)

print(core_dependency[1])
print(qualifying_child[1])
print(qualifying_relative[1])
print(f"""
Tim fails the qualifying child test but passes the qualifying relative test.
The Smiths may be able to claim Tim as a qualifying relative dependent.""")

- **b) Assume the original facts except that Tim earned 10,000 and used all the funds for his own support. Are the Smiths able to claim Tim as a dependent?**

In [ ]:
tim_own_support = 10000

print("A change in income won't change the result of the qualifying child test, so let's re-run the qualifying relative test:")

qualifying_relative = dependency_test.qualifying_relative_test(
    relationship_test=True, # A sibling of the taxpayer's mother of father
    support_test=provided_support < own_support, 
    gross_income_test=tim_own_support < gross_income_limit)

print(qualifying_relative[1])
print(f"""\nTim's income of ${tim_own_support:,.0f} exceeds the gross income limit for 2024 of ${gross_income_limit:,.0f},
so he fails that test. He also fails the support test, his own contribution now exeeds half of his
total support, meaning the Smiths no longer provide more than half. Failing eiteher test alone disqualifies him,
but here both fail. The Smiths may not claim uncle Tim as qualifying relative dependent.""")


- c) **Assume the original facts except that Tim is a friend of the family and not John’s uncle.**

In [ ]:
print("""Assuming the original facts, the Smiths would not be able to claim a Tim a qualifying relative dependent.

Here is why:

A person meets the qualifying relative member of the household test if that person has the same principal place of 
abode as the taxpayer for the entire year. This is true even if the person does not have a qualifying family relationship 
with the taxpayer.

The rules require Tim to live with the Smiths the whole year, but he moved in in March, failing the test.""")

- d) **Assume the original facts except that Tim is a friend of the family and not John’s uncle and Tim lived with the Smiths for the entire year.**

In [ ]:
print("""Assuming the original facts, the Smiths may be able to claim Tim as a qualifying relative dependent.

Here is why:

A person meets the qualifying relative member of the household test if that person has the same principal place of 
abode as the taxpayer for the entire year. This is true even if the person does not have a qualifying family relationship 
with the taxpayer.

The rules require Tim to live with the smiths the whole year and he did, so he passed the test.""")

---

##### **Francine’s mother Donna and her father Darren separated and divorced in September of this year. Francine lived with both parents until the separation. Francine does not provide more than half of her own support. Francine is 15 years old at the end of the year.**

- **a) Is Francine a qualifying child to Donna?**
- **b) Is Francine a qualifying child to Darren?**

In [ ]:
dependent_age = 15   # Given
parents_age = 45     # Not given, seems reasonable
start_date = date(2024, 1, 1)
end_date = date(2024, 9, 1)

core_dependency = dependency_test.core_dependency_test(
    dependent_taxpayer_test=True,       # Daughter
    citizenship_residency_test=True,    # Not given, assuming True
    joint_return_test=True              # Not given, assuming True
    )

qualifying_child = dependency_test.qualifying_child_test(
    relationship_test=True,                                                 # Not a descendant, sibling or descendant of sibling
    age_test=dependent_age < 19 and dependent_age < parents_age,            # 15 < 19 and 15 < 45 = true
    residence_test= dependency_test.residence_tests(start_date, end_date),  # Given
    half_support_test=True)                                                 # True as given

print(qualifying_child[1])
print("\nSince Francine passes the qualifying child test, she is considered a qualifying child for both parents.")
print("Tie-breaking rules are considered in the next questions.")


- **c) Assume Francine spends more time living with Darren than Donna after the separation. Who may claim Francine as a dependent?**

In [ ]:

print("""The tie-breaking rules state that the parent with whom the child has resided for the longest period 
of time during the year has priority for claiming the person as a dependent.  Darren gets the claim over Francine as a dependent.""")

- **d) Assume Francine spends an equal number of days with her mother and her father and that Donna has AGI of 52,000 and Darren has AGI of 50,000. Who may claim Francine as a dependent?**

In [ ]:
print("""The tie-breaking rules state that if the child resides with each parent for equal amounts of time during the year, 
the child is a qualifying child of the parent with the higher AGI.  Accordingly, Donna gets to claim Francine as a dependent.""")

---

##### **Jamel and Jennifer have been married 30 years and have filed a joint return every year of their marriage. Their three daughters, Jade, Lindsay, and Abbi, are ages 12, 17, and 22, respectively, and all live at home. None of the daughters provides more than half of her own support. Abbi is a full-time student at a local university and does not have any gross income.**

- **a) Which, if any, of the daughters qualify as dependents of Jamel and Jennifer?**

In [ ]:
# Let's aggregate all three daughters into on example.

core_dependency = dependency_test.core_dependency_test(
    dependent_taxpayer_test=True,       # Daughters, qualifyied child
    citizenship_residency_test=True,    # Not given, assuming True
    joint_return_test=True)             # Not given, assuming True
    

qualifying_child = dependency_test.qualifying_child_test(
    relationship_test=True,             # Not a descendant, sibling or descendant of sibling
    age_test=True,                      # All daughters are below 19, Abbie is below 24 and full-time student
    residence_test= True,               # Not given, assuming True
    half_support_test=True)             # True, per text

print(core_dependency[1])
print(qualifying_child[1])
print(f"\nGiven the facts provided, all daughters qualify as qualifying child dependents to Jamel and Jennifer.")

- **b) Assume the original facts except that Abbi is married. She and her husband live with Jamel and Jennifer while attending school and they file a joint return. Abbi and her husband reported a 1,000 tax liability on their tax return. If all parties are willing, can Jamel and Jennifer claim Abbi as a dependent on their tax return? Why or why not?**

In [ ]:
core_dependency = dependency_test.core_dependency_test(
    dependent_taxpayer_test=True,       # Daughter, qualifying child
    citizenship_residency_test=True,    # Not given, assuming True
    joint_return_test=False)            # Married filing jointly and tax liability

print(core_dependency[1])
print("""\nAbbie cannot be claimed as either a qualifying child or qualifying relative due to the fact that she filed a 
joint return with her husband and have a tax liability of 1,000, failing the Joint Return Test.""")

- **c) Assume the same facts as part (b), except that Abbi and her husband report a 0 tax liability on their joint tax return. Also, if the couple had filed separately, Abbi would not have had a tax liability on her return, but her husband would have had a 250 tax liability on his separate return. Can Jamel and Jennifer claim Abbi as a dependent on their tax return? Why or why not?**

In [ ]:
core_dependency = dependency_test.core_dependency_test(
    dependent_taxpayer_test=True,       # Daughters, qualifying child
    citizenship_residency_test=True,    # Not given, assuming True
    joint_return_test=False)            # Husban has tax liability of $250

print(core_dependency[1])
print("""
In order to pass the joint return test an individual Must not file a joint return with their spouse 
unless there is no tax liability on the couple’s joint return and there would not have been any tax liability 
on either spouse’s tax return if they had filed separately.  Given the facts, her husband has a tax liability of
$250, so she fails the joint return test.""")

- **d) Assume the original facts except that Abbi is married. Abbi files a separate tax return. Abbi’s husband files a separate tax return and reports a 250 tax liability on the return. Can Jamel and Jennifer claim Abbi as a dependent?**

In [ ]:
print("""
Same as above.  It does not matter if they are filing jointly or separately. In order to pass the joint return 
test an individual Must not file a joint return with their spouse unless there is no tax liability on the couple’s 
joint return and there would not have been any tax liability on either spouse’s tax return if they had filed separately.  
Given the facts, her husband has a tax liability of $250, so she fails the joint return test.""")

---

##### **Dean Kastner is 78 years old and lives by himself in an apartment in Chicago. Dean’s gross income for the year is $2,500. Dean’s support is provided as follows: himself (5 percent), his daughters Camille (25 percent) and Rachel (30 percent), his son Zander (5 percent), his friend Frankie (15 percent), and his niece Sharon (20 percent).**

In [88]:
support = {
    "Himself": [.05, False],
    "Camille": [.25, True],
    "Rachel": [.30, True],
    "Zander": [.05, True],
    "Frankie": [.15, False],
    "Sharon": [.20, True]
}

- **a) Absent a multiple support agreement, of the parties mentioned in the problem, who may claim Dean as a dependent?**

In [90]:
print("""In order to claim an individual the taxpayer must meet the support test, which test whether the taxpayer provided more than
half of the support to the individual.  
      
Does anyone individually provided more than 50% of support?\n""")

result = False

for k, v in support.items():
    if v[0] > .50:
        print(f"\t{k}: {v[0]*100}%")
        result = True
    else:
        print(f"\t{k}: {v[0]*100}% - Does not exceed 50% support.")

if result == False:
    print("\nAbsent of a support agreement, nobody satifies the support test on their own.")

In order to claim an individual the taxpayer must meet the support test, which test whether the taxpayer provided more than
half of the support to the individual.  

Does anyone individually provided more than 50% of support?

	Himself: 5.0% - Does not exceed 50% support.
	Camille: 25.0% - Does not exceed 50% support.
	Rachel: 30.0% - Does not exceed 50% support.
	Zander: 5.0% - Does not exceed 50% support.
	Frankie: 15.0% - Does not exceed 50% support.
	Sharon: 20.0% - Does not exceed 50% support.

Absent of a support agreement, nobody satifies the support test on their own.


- **b) Under a multiple support agreement, who is eligible to claim Dean as a dependent? Explain.**

In [99]:
print(f"""Multiple support agreement rule requires:
      
    1) No single person provided more that 50% percent of support
    2) The group together provided more than 50%
    3) Anyone who wants to claim mus have contributed more than 10%
    4) Still needs to pass the relationship test (or member-of-household test)
""")

print("""Test 1: Per the previous question, no one contributed more than 50% support""")

total_support = 0
for k, v in support.items():
    if k != "Himself":
        total_support += v[0]

print(f"""
Test 2: The group together provided more than 50% support:
    Support of the group: {total_support*100:,.0f}
    Results: {"Passed" if total_support > .50 else "Failed"}
""")

def ten_percent_test(dict):
    for k, v in dict.items():
        if v[0] > .10:
            print(f"\t{k}: {v[0]*100:.0f} - Contributed more than 10%")

print(f"""Test 3: Anyone who wants to claim mus have contributed more than 10%""")
ten_percent_test(support)
print("\tAll four pass this test.")

qualifying_relatives = []

def relationship_test(dict):
    for k, v in dict.items():
        if v[0] > .10 and v[1] == True:
            qualifying_relatives.append(k)

relationship_test(support)

print(f"""
Test 4: Must pass the relationship test (or member-of-household test)
      The following people pass the relationship test:""")

for x in qualifying_relatives:
    print(f"\t- {x}")

print("""
Conclussion:
    Any one of Camille, Rachel, or Sharon may claim Dean, als long as the other two sign
    a written declaration (Form 2120) agreeing not claim him.
      
      """)

Multiple support agreement rule requires:

    1) No single person provided more that 50% percent of support
    2) The group together provided more than 50%
    3) Anyone who wants to claim mus have contributed more than 10%
    4) Still needs to pass the relationship test (or member-of-household test)

Test 1: Per the previous question, no one contributed more than 50% support

Test 2: The group together provided more than 50% support:
    Support of the group: 95
    Results: Passed

Test 3: Anyone who wants to claim mus have contributed more than 10%
	Camille: 25 - Contributed more than 10%
	Rachel: 30 - Contributed more than 10%
	Frankie: 15 - Contributed more than 10%
	Sharon: 20 - Contributed more than 10%
	All four pass this test.

Test 4: Must pass the relationship test (or member-of-household test)
      The following people pass the relationship test:
	- Camille
	- Rachel
	- Sharon

Conclussion:
    Any one of Camille, Rachel, or Sharon may claim Dean, als long as the other 

- **c) Assume that Camille is allowed to claim Dean as a dependent under a multiple support agreement. Camille is single, and Dean is her only dependent. What is Camille’s filing status?**

In [100]:

print(f"""
Camille's filling status is: Single
      
In order to qualify as Head of Household she must provide more that half of support of the cost of maintaining Dean's household.
Given tha she only paid 25% of support, she does not qualify of Head of Household status.""")


Camille's filling status is: Single

In order to qualify as Head of Household she must provide more that half of support of the cost of maintaining Dean's household.
Given tha she only paid 25% of support, she does not qualify of Head of Household status.


---